# Exploratory Data Analysis — DataFrame

Generated by `flashkeras.notebooks`.

Edit the **Parameters** cell below with your CSV path and re-run everything (`Run All`).

In [ ]:
csv_path = "REPLACE_ME.csv"
target_column = None  # optional: name of the column you want to predict
separator = ","


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (10, 5)


## 2. Load data

In [ ]:
df = pd.read_csv(csv_path, sep=separator)
print(f"Shape: {df.shape}")
df.head()


## 3. Overview

In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


## 4. Missing values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print("No missing values found.")
else:
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    display(missing_df)
    missing_df["missing_pct"].plot(kind="barh", title="Missing values (%)")
    plt.gca().invert_yaxis()
    plt.show()


## 5. Duplicates

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes / len(df) * 100:.2f}%)")


## 6. Column types

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
datetime_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()

print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Datetime ({len(datetime_cols)}): {datetime_cols}")


## 7. Numeric distributions

In [ ]:
if numeric_cols:
    df[numeric_cols].hist(figsize=(14, 10), bins=30)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns found.")


## 8. Categorical value counts (top columns)

In [ ]:
for col in categorical_cols[:10]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(10))


## 9. Correlation matrix (numeric)

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticklabels(corr.columns)
    fig.colorbar(im)
    plt.title("Correlation matrix")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for a correlation matrix.")


## 10. Numeric outlier detection

The IQR rule flags values below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR. Review the flagged percentage rather than treating every flagged value as an error.

In [ ]:
outlier_rows = []

for col in numeric_cols:
    values = df[col].dropna()
    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    outlier_count = int(outlier_mask.sum())

    outlier_rows.append({
        "column": col,
        "outlier_count": outlier_count,
        "outlier_pct": round(outlier_count / len(df) * 100, 2),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
    })

if outlier_rows:
    outlier_summary = pd.DataFrame(outlier_rows).sort_values("outlier_count", ascending=False)
    display(outlier_summary)

    columns_with_outliers = outlier_summary.loc[
        outlier_summary["outlier_count"] > 0, "column"
    ].tolist()
    if columns_with_outliers:
        df[columns_with_outliers].plot(kind="box", subplots=True, layout=(-1, 2), figsize=(12, 4))
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric outliers found using the IQR rule.")
else:
    print("No numeric columns available for outlier detection.")

## 11. Target column analysis (if set)

In [ ]:
if target_column is not None:
    print(f"Target: {target_column}")
    display(df[target_column].describe())

    if target_column in categorical_cols or df[target_column].nunique() < 20:
        df[target_column].value_counts().plot(kind="bar", title=f"Distribution of {target_column}")
        plt.show()
    else:
        df[target_column].plot(kind="hist", bins=30, title=f"Distribution of {target_column}")
        plt.show()
else:
    print("Set `target_column` in the Parameters cell to analyze it here.")


## 12. Target balance (if set)

For classification targets, compare class percentages. A large difference between the most and least common classes may indicate imbalance; the appropriate threshold depends on the problem and evaluation metric.

In [ ]:
if target_column is None:
    print("Set `target_column` in the Parameters cell to check class balance.")
else:
    target_counts = df[target_column].value_counts(dropna=False)
    target_percentages = (target_counts / len(df) * 100).round(2)
    balance_df = pd.DataFrame({
        "count": target_counts,
        "percentage": target_percentages,
    })
    display(balance_df)

    balance_df["percentage"].plot(kind="bar", title=f"Class distribution: {target_column}")
    plt.ylabel("Percentage of rows")
    plt.xlabel(target_column)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    if len(balance_df) > 1:
        majority_pct = balance_df["percentage"].max()
        minority_pct = balance_df["percentage"].min()
        imbalance_ratio = majority_pct / minority_pct if minority_pct > 0 else np.inf
        print(f"Largest class: {majority_pct:.2f}%")
        print(f"Smallest class: {minority_pct:.2f}%")
        print(f"Majority-to-minority ratio: {imbalance_ratio:.2f}:1")
        if imbalance_ratio >= 2:
            print("The target appears imbalanced; consider stratified splits and class-aware metrics.")
        else:
            print("No strong imbalance detected by the 2:1 ratio heuristic.")
    else:
        print("The target has fewer than two distinct classes.")

## Next steps

- Handle missing values (impute or drop)
- Encode categorical columns
- Check for outliers in numeric columns
- If this is heading toward a Keras model, check out `flashkeras.notebooks.new_notebook("image_classification_baseline")` or `"text_classification_baseline"` depending on your data type.